# R1-07 README + submit
Outputs: `README.md`, `requirements.txt`, `.env.example` (repo root).
Cells: 1 Check, 2 Backup, 3 Create, 4 Verify, 5 Unchanged, 6 Undo (RUN_UNDO = False), 7 Git (print only).
The README has no invented numbers: the Tableau link is read from dashboard/dashboard_documentation.md at run time, all figures stay in their source files.
The portal submission (item 4) is a manual step by Ayhan: set SUBMITTED_ON_PORTAL = True in cell 1 after submitting.

In [1]:
# 1 Check: right folder, input files exist
from pathlib import Path
import re, json, sys, hashlib, shutil
from datetime import datetime
from urllib.parse import quote, unquote

SUBMITTED_ON_PORTAL = False   # Ayhan sets True after submitting as project-5 on the portal

REPO = Path.cwd() if (Path.cwd() / 'AGENTS.md').exists() else Path.cwd().parent
NL = chr(10)

def ok(name, cond):
    print('PASS' if cond else 'FAIL', '-', name)
    return bool(cond)

OUTPUTS = ['README.md', 'requirements.txt', '.env.example']
INPUTS = ['AGENTS.md', 'board.md', 'specs/R1-07_submit.md',
          'dashboard/dashboard_documentation.md', 'n8n/workflow.json', 'n8n/workflow_documentation.md',
          'n8n/airtable_setup.md', 'n8n/airtable_seed.csv', 'langsmith/monitoring_note.md',
          'cost_estimation/cost_analysis.md', 'research/sector_research.md', 'research/opportunities_risks.md',
          'research/use_cases.md', 'data/README.md', 'decisions/decision_log.md', 'notebooks/R1-05_langsmith.ipynb']
ok('repo folder has AGENTS.md', (REPO / 'AGENTS.md').exists())
for p in INPUTS:
    ok('input exists: ' + p, (REPO / p).exists())
print('SUBMITTED_ON_PORTAL =', SUBMITTED_ON_PORTAL)

PASS - repo folder has AGENTS.md
PASS - input exists: AGENTS.md
PASS - input exists: board.md
PASS - input exists: specs/R1-07_submit.md
PASS - input exists: dashboard/dashboard_documentation.md
PASS - input exists: n8n/workflow.json
PASS - input exists: n8n/workflow_documentation.md
PASS - input exists: n8n/airtable_setup.md
PASS - input exists: n8n/airtable_seed.csv
PASS - input exists: langsmith/monitoring_note.md
PASS - input exists: cost_estimation/cost_analysis.md
PASS - input exists: research/sector_research.md
PASS - input exists: research/opportunities_risks.md
PASS - input exists: research/use_cases.md
PASS - input exists: data/README.md
PASS - input exists: decisions/decision_log.md
PASS - input exists: notebooks/R1-05_langsmith.ipynb
SUBMITTED_ON_PORTAL = False


In [2]:
# 2 Backup: copy the 3 output files (if they exist) to _backup/<timestamp>/, hash every other file
TS = datetime.now().strftime('%Y%m%d_%H%M%S')
BK = REPO / '_backup' / TS
EXISTED = {}
for p in OUTPUTS:
    EXISTED[p] = (REPO / p).exists()
    if EXISTED[p]:
        (BK / p).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(REPO / p, BK / p)

# files we do not own: skipped are .git, _backup, logs (hooks write there), board.md, this notebook, the 3 outputs, .env (never read)
SKIP_DIRS = {'.git', '_backup', '.ipynb_checkpoints', 'logs'}
OWN = set(OUTPUTS) | {'board.md', 'notebooks/R1-07_submit.ipynb', '.env'}

def snapshot():
    snap = {}
    for p in REPO.rglob('*'):
        if not p.is_file():
            continue
        rel = p.relative_to(REPO).as_posix()
        if set(rel.split('/')) & SKIP_DIRS or rel in OWN or p.name == '.DS_Store':
            continue
        snap[rel] = hashlib.sha256(p.read_bytes()).hexdigest()
    return snap

HASH_BEFORE = snapshot()
print('Backup folder:', BK)
print('Existed before:', EXISTED)
print('Files hashed (not ours):', len(HASH_BEFORE))

Backup folder: /Users/macbook-air/Desktop/AI consulting/Week8/Day4/ecommerce_customersatisfaction_adiadoption/_backup/20261127_170132
Existed before: {'README.md': True, 'requirements.txt': True, '.env.example': True}
Files hashed (not ours): 81


In [3]:
# 3 Create: README.md, requirements.txt (add missing packages only), .env.example (names only)

# --- facts read at run time ---
doc = (REPO / 'dashboard/dashboard_documentation.md').read_text(encoding='utf-8')
m = re.search(r'Tableau Public link: (https?://[^ ]+)', doc)
TABLEAU = m.group(1) if m else ''
TWBX = sorted((REPO / 'dashboard').glob('*.twbx'))[0].name
SHOT = sorted((REPO / 'dashboard/screenshots').glob('*.png'))[0].name
print('Tableau link:', TABLEAU)

# --- .env.example: variable names found in notebooks/ and n8n/ ---
NAME_RE = r'(?<![A-Za-z0-9_])((?:OPENAI|LANGSMITH|LANGCHAIN|AIRTABLE|N8N)_[A-Z_]+)(?![A-Za-z0-9_])'
NOT_ENV = {'LANGSMITH_SHARE_URL', 'N8N_SYSTEM', 'N8N_USER', 'N8N_LLM'}   # plain notebook variables, not env names
files = list((REPO / 'notebooks').glob('*.ipynb')) + list((REPO / 'n8n').glob('*.json')) + list((REPO / 'n8n').glob('*.md'))
FOUND = set()
for f in files:
    FOUND |= set(re.findall(NAME_RE, f.read_text(encoding='utf-8')))
FOUND -= NOT_ENV
print('Env names found in notebooks/ and n8n/ (check for junk names):', sorted(FOUND))

envx = REPO / '.env.example'
lines = envx.read_text(encoding='utf-8').split(NL) if envx.exists() else []
have = {l.split('=')[0].strip() for l in lines if '=' in l and not l.startswith('#')}
missing = sorted(FOUND - have)
if missing or not envx.exists():
    lines = [l for l in lines if l != ''] + [n + '=' for n in missing]
    envx.write_text(NL.join(lines) + NL, encoding='utf-8')
print('Names added to .env.example:', missing)

# --- requirements.txt: add only third-party packages that notebooks really import ---
# handles: import a / import a as b / import a, b / import a.b as c / from a.b import c
IMPORT_PART = r'[A-Za-z_][A-Za-z0-9_.]*([ ]+as[ ]+[A-Za-z_][A-Za-z0-9_]*)?'
FROM_RE = r'^[ ]*from[ ]+([A-Za-z_][A-Za-z0-9_]*)[A-Za-z0-9_.]*[ ]+import[ ]+'
PKG = {'dotenv': 'python-dotenv'}   # import name -> pip name
mods = set()
for nb in (REPO / 'notebooks').glob('*.ipynb'):
    for c in json.loads(nb.read_text(encoding='utf-8'))['cells']:
        if c['cell_type'] != 'code':
            continue
        for line in ''.join(c['source']).split(NL):
            s = line.strip()
            if s.startswith('import '):
                for part in s[7:].split(','):
                    part = part.strip()
                    if re.fullmatch(IMPORT_PART, part):
                        mods.add(part.split('.')[0].split(' ')[0])   # top-level package name
            else:
                mm = re.match(FROM_RE, line)
                if mm:
                    mods.add(mm.group(1))
third = sorted(PKG.get(x, x) for x in mods if x not in sys.stdlib_module_names)
print('Third-party imports in notebooks:', third)

req = REPO / 'requirements.txt'
old = req.read_text(encoding='utf-8').split(NL) if req.exists() else []
old = [l for l in old if l.strip() != '']
def norm(l):
    return re.split('[<>=~! ]', l.strip())[0].lower().replace('_', '-')
listed = {norm(l) for l in old}
add = [p for p in third if p.lower().replace('_', '-') not in listed]
if add or not req.exists():
    req.write_text(NL.join(old + add) + NL, encoding='utf-8')   # existing lines kept, new ones appended
print('Packages added to requirements.txt:', add)

# --- README.md ---
R = []
R += ['# Spreekorb: AI for email customer service (Ironhack capstone, Round 1)', '']
R += ['Independent case study. Spreekorb is an invented Berlin e-commerce company (about 500 employees, no AI yet). All data is public or synthetic and is labelled as such.', '']
R += ['## Overview', '']
R += ['- Client: Spreekorb. CEO Chleo fears that AI is a black box. This project answers with transparency: every number has a source or a label (estimate, measured, synthetic), every AI draft shows a short reason and the policy sections it used, and a person approves every draft. Nothing is sent automatically.']
R += ['- Focus: email customer service, SLA (first response time) and CSAT (customer satisfaction).']
R += ['- UC1: email triage plus reply drafts, a human approves (built as the Round 1 POC in n8n).']
R += ['- UC2: root cause report to the business (described in research/use_cases.md).']
R += ['- UC3: team-level view, a coaching signal and not a scorecard (team tile M5 in the dashboard).']
R += ['- Round 1 deliverables: research pack, data prep, Tableau dashboard, n8n POC, LangSmith monitoring sample, cost and timeline, this README.', '']
R += ['## Data and privacy', '']
R += ['- Data is synthetic or public: Shopzilla (Kaggle) and Tobi-Bueck (Hugging Face, synthetic) are proxy datasets, the Spreekorb emails and the policy documents are invented. Sources are listed in `data/README.md` and `dashboard/dashboard_documentation.md`.']
R += ['- Team level only, minimum group of 5. No agent names or IDs are recorded.']
R += ['- Before real use: data processing agreement (DPA), works council (Betriebsrat) agreement and a data protection impact assessment (DPIA), see `n8n/workflow_documentation.md`.']
R += ['- No keys in the repo. Keys live in a local `.env` file that is git-ignored; `.env.example` lists the names only.', '']
R += ['## What is where', '']
R += ['| Path | Content |', '|---|---|']
R += ['| `research/` | Sector research (`research/sector_research.md`), opportunities and risks (`research/opportunities_risks.md`), use cases (`research/use_cases.md`) |']
R += ['| `data/` | Data README (`data/README.md`), processed data in `data/processed/`, timing log (`data/timing_log.csv`); raw downloads are not committed |']
R += ['| `dashboard/` | Tableau workbook: [' + TWBX + '](dashboard/' + quote(TWBX) + '), documentation with the Tableau Public link (`dashboard/dashboard_documentation.md`), metric CSVs in `dashboard/data/` |']
R += ['| `n8n/` | POC workflow (`n8n/workflow.json`, credentials removed), documentation (`n8n/workflow_documentation.md`), Airtable setup (`n8n/airtable_setup.md`), seed emails (`n8n/airtable_seed.csv`), prompt (`n8n/triage_prompt.md`), screenshots in `n8n/screenshots/` |']
R += ['| `langsmith/` | Monitoring sample: note (`langsmith/monitoring_note.md`), trace summary (`langsmith/trace_summary.csv`), human review (`langsmith/human_review.csv`), screenshots in `langsmith/screenshots/` |']
R += ['| `cost_estimation/` | Cost analysis (`cost_estimation/cost_analysis.md`) and timeline (`cost_estimation/timeline_estimate.md`) |']
R += ['| `knowledgebase/` | Invented Spreekorb policy documents (PDF) used by the draft prompt |']
R += ['| `decisions/` | Decision log (`decisions/decision_log.md`) |']
R += ['| feedback | Round 1 decision (added by card R1-08) |']
R += ['| `notebooks/` | One task notebook per card (build evidence, run top to bottom) |']
R += ['| `specs/`, `board.md`, `logs/` | How the project was built: specs, board, agent log |', '']
R += ['## How to view the dashboard', '']
R += ['1. Open the Tableau Public link: ' + TABLEAU]
R += ['2. Or open the packaged workbook [' + TWBX + '](dashboard/' + quote(TWBX) + ') in Tableau Desktop or Tableau Public.']
R += ['3. Each tile has a headline, a source footer and a SYNTHETIC badge where the numbers are made up. The metric definitions are in `dashboard/dashboard_documentation.md`.']
R += ['4. No Tableau? See the screenshot: ![Dashboard](dashboard/screenshots/' + quote(SHOT) + ')', '']
R += ['## How to run the POC', '']
R += ['Needs: Python 3.12, an OpenAI key, a LangSmith key, an n8n account and an Airtable account.', '']
R += ['1. Install and set keys: `pip install -r requirements.txt`, copy `.env.example` to `.env` and fill in the values. Variable names: ' + ', '.join(sorted(have | set(missing))) + '. Never commit `.env`.']
R += ['2. Airtable: create the base and the Review queue table as described in `n8n/airtable_setup.md`.']
R += ['3. n8n: import `n8n/workflow.json` (Workflows > Import from file), paste the Airtable base id and table id, select your Airtable and OpenAI credentials, activate the workflow.']
R += ['4. Import `n8n/airtable_seed.csv` into the table. Each row gets an AI draft and the status Pending review. A person then sets Approved, Edited or Rejected. Nothing is sent. Details: `n8n/workflow_documentation.md`.']
R += ['5. LangSmith: run `notebooks/R1-05_langsmith.ipynb` top to bottom (uses OPENAI_API_KEY and LANGSMITH_API_KEY). It writes `langsmith/trace_summary.csv` and `langsmith/monitoring_note.md`.']
R += ['6. Optional: re-run the other notebooks in `notebooks/` in card order (R1-02a downloads the raw data first).', '']
R += ['## About the numbers', '']
R += ['This README quotes no figures. Every figure (CSAT, SLA, timing, cost, payback) lives in its source file with its label (estimate, measured or synthetic): `dashboard/dashboard_documentation.md`, `langsmith/monitoring_note.md`, `cost_estimation/cost_analysis.md`.']
README = NL.join(R) + NL
(REPO / 'README.md').write_text(README, encoding='utf-8')
print('Written:', OUTPUTS)

Tableau link: https://public.tableau.com/app/profile/ayhan.demir/viz/SpreekorbemailCSAIdraftshuman-approved/Dashboard1#2

Workbook:
Env names found in notebooks/ and n8n/ (check for junk names): ['AIRTABLE_ID', 'LANGSMITH_API_KEY', 'LANGSMITH_PROJECT', 'LANGSMITH_TRACING', 'OPENAI_API_KEY']
Names added to .env.example: ['AIRTABLE_ID']
Third-party imports in notebooks: ['datasets', 'kagglehub', 'langsmith', 'numpy', 'openai', 'pandas', 'pypdf', 'python-dotenv']
Packages added to requirements.txt: ['numpy']
Written: ['README.md', 'requirements.txt', '.env.example']


In [4]:
# 4 Verify: one PASS/FAIL per acceptance item
readme = (REPO / 'README.md').read_text(encoding='utf-8')

# ---- Item 1: README sections, terms, links ----
print('== Item 1: README')
heads = [l[3:].strip().lower() for l in readme.split(NL) if l.startswith('## ')]
for h in ['overview', 'what is where', 'how to view the dashboard', 'how to run the poc']:
    ok('README section: ' + h, h in heads)
for t in ['Spreekorb', 'UC1', 'UC2', 'UC3', 'black box', 'SLA', 'CSAT', 'synthetic', 'Team level', 'minimum group of 5', '.env.example']:
    ok('README mentions: ' + t, t in readme)
ok('Tableau Public link in README', TABLEAU != '' and TABLEAU in readme)

paths = set()
for t in re.findall('`([A-Za-z0-9_./-]+)`', readme):
    if '/' in t or t in ('requirements.txt', 'board.md', '.env.example'):
        paths.add(t)
for t in re.findall(r'[]][(]([^)]+)[)]', readme):
    if not t.startswith('http'):
        paths.add(unquote(t))
bad = sorted(p for p in paths if not (REPO / p).exists())
ok('every repo path in README exists (' + str(len(paths)) + ' checked)', not bad)
if bad:
    print('  missing:', bad)

# ---- Item 2: no secrets ----
print('== Item 2: no secrets')
HARD = {'OpenAI key': r'(?<![A-Za-z0-9])sk-[A-Za-z0-9_-]{20,}', 'LangSmith key': r'lsv2_[A-Za-z0-9_]{16,}',
        'Airtable token': r'pat[A-Za-z0-9]{14}[.][A-Za-z0-9]{30,}', 'JWT (n8n API key)': r'eyJ[A-Za-z0-9_-]{10,}[.][A-Za-z0-9_-]{10,}[.][A-Za-z0-9_-]{10,}',
        'Bearer token': r'Bearer[ ]+[A-Za-z0-9._-]{30,}'}
Q = chr(39) + chr(34)
SOFT = r'(?i)(api[_-]?key|token|secret|password)[ ]*[=:][ ]*[' + Q + ']?[A-Za-z0-9+/_-]{32,}'
AIRTABLE_ID = r'(?<![A-Za-z0-9])(app|tbl)[A-Za-z0-9]{14}(?![A-Za-z0-9])'
SUFFIX = {'.md', '.py', '.json', '.csv', '.txt', '.ipynb', '.example', '.yml', '.yaml', '.toml', '.cfg', ''}
hard_hits, soft_hits, id_hits, n_scanned = [], [], [], 0
for p in REPO.rglob('*'):
    if not p.is_file() or p.name == '.env':
        continue
    rel = p.relative_to(REPO).as_posix()
    if set(rel.split('/')) & {'.git', '_backup', '.ipynb_checkpoints'} or rel.startswith('data/raw/'):
        continue
    if p.suffix not in SUFFIX or p.stat().st_size > 20_000_000:
        continue
    text = p.read_text(encoding='utf-8', errors='ignore')
    n_scanned += 1
    for name, rx in HARD.items():
        if re.search(rx, text):
            hard_hits.append((rel, name))
    if re.search(SOFT, text):
        soft_hits.append(rel)
    if re.search(AIRTABLE_ID, text):
        id_hits.append(rel)
ok('no key patterns in ' + str(n_scanned) + ' text files', not hard_hits)
if hard_hits:
    print('  hits:', hard_hits)
if soft_hits:
    print('WARN - look-alike key=value lines (check by eye):', soft_hits)
if id_hits:
    print('WARN - Airtable-style base/table ids (not secret keys, but check):', id_hits)
extra = [f.name for f in (REPO / 'n8n').glob('*.json') if f.name != 'workflow.json']
if extra:
    print('WARN - live n8n export is committed (commit 7e312e9):', extra)
    print('       It has n8n credential references (id and name only, no keys) and one real Airtable base id + table id.')
    print('       Ayhan: decide to untrack or remove it before submission. This notebook does not touch it.')
gi = [l.strip() for l in (REPO / '.gitignore').read_text(encoding='utf-8').split(NL)]
ok('.env is listed in .gitignore', '.env' in gi)
okvals = True
for l in (REPO / '.env.example').read_text(encoding='utf-8').split(NL):
    if '=' in l and not l.startswith('#'):
        val = l.split('=', 1)[1].strip()
        if val not in ('', 'true', 'false', 'spreekorb-capstone') and not val.startswith('your_'):
            okvals = False
            print('  suspicious value for', l.split('=')[0])
ok('.env.example has names only (empty or placeholder values)', okvals)
ok('.env.example has the names found in notebooks/ and n8n/', FOUND <= {l.split('=')[0] for l in (REPO / '.env.example').read_text(encoding='utf-8').split(NL) if '=' in l})
ok('requirements.txt has every third-party import', all(p.lower().replace('_', '-') in {norm(l) for l in (REPO / 'requirements.txt').read_text(encoding='utf-8').split(NL) if l.strip()} for p in third))

# ---- Item 3: Round 1 checklist ----
print('== Item 3: Round 1 checklist')
def exists(p):
    if '*' in p:
        return bool(list(REPO.glob(p)))
    if (REPO / p).exists():
        return True
    q = Path(p)
    if q.parent.as_posix() == 'notebooks':   # notebook names may carry a suffix, e.g. _AyhanDemir
        return bool(list((REPO / 'notebooks').glob(q.stem + '*' + q.suffix)))
    return False

# look for a real checklist file or text
cands = []
for d in ['research', 'specs', 'decisions', 'feedback']:
    if (REPO / d).exists():
        cands += [p for p in (REPO / d).rglob('*') if p.suffix in ('.md', '.txt')]
cands += [REPO / 'board.md'] + [p for p in REPO.glob('*') if p.suffix in ('.md', '.txt', '.pdf') and p.name not in ('README.md', 'AGENTS.md', 'CLAUDE.md')]
cl_hits = []
for p in cands:
    if p.name == 'R1-07_submit.md' or not p.is_file() or p.suffix == '.pdf':
        continue
    if 'checklist' in p.name.lower() or 'checklist' in p.read_text(encoding='utf-8', errors='ignore').lower():
        cl_hits.append(p)
items = []   # (label, path)
if cl_hits:
    print('Checklist candidates:', [p.relative_to(REPO).as_posix() for p in cl_hits])
    for p in cl_hits:
        for l in p.read_text(encoding='utf-8', errors='ignore').split(NL):
            if re.match('^[ ]*[-*][ ]*[[][ xX][]]', l):
                toks = re.findall('[A-Za-z0-9_./-]+[.][A-Za-z]{2,5}', l)
                items.append((l.strip(), toks[0] if toks else None))
if not items:
    print('NOTE: Round 1 checklist source was NOT found (no checklist file or checkbox list in research/, specs/, decisions/, board.md, repo root).')
    print('NOTE: list built from board.md R1-00..R1-06 evidence + key outputs. Ayhan: please confirm against the real checklist.')
    board = (REPO / 'board.md').read_text(encoding='utf-8')
    for row in board.split(NL):
        c = [x.strip() for x in row.strip().strip('|').split('|')]
        if len(c) >= 5 and re.match('R1-0[0-6]', c[0]):
            ev = c[4].split(' ')[0]
            items.append((c[0] + ' ' + c[1] + ' (board status ' + c[3] + ')', ev))
            if c[3] != 'Approved':
                ok(c[0] + ' board status is Approved', False)
    KEY = ['research/sector_research.md', 'research/opportunities_risks.md', 'research/use_cases.md',
           'data/README.md', 'data/processed/shopzilla_email_clean.csv', 'data/processed/tobi_tuning_150.csv',
           'data/processed/tobi_test_50.csv', 'data/processed/spreekorb_emails_60.csv', 'data/timing_log.csv',
           'dashboard/*.twbx', 'dashboard/dashboard_documentation.md', 'dashboard/screenshots/*.png',
           'n8n/workflow.json', 'n8n/workflow_documentation.md', 'n8n/airtable_setup.md', 'n8n/airtable_seed.csv',
           'n8n/screenshots/workflow.png', 'n8n/screenshots/airtable_review.png',
           'langsmith/monitoring_note.md', 'langsmith/trace_summary.csv', 'langsmith/human_review.csv', 'langsmith/screenshots/*.png',
           'cost_estimation/cost_analysis.md', 'cost_estimation/timeline_estimate.md']
    items += [('key output', k) for k in KEY]
    ok('Tableau Public link present in dashboard documentation', TABLEAU != '')
all3 = True
for label, p in items:
    if p is None:
        print('MANUAL - no file named in item, Ayhan to tick:', label)
    else:
        all3 &= ok('evidence exists: ' + p + '  <- ' + label, exists(p))
ok('Item 3: all R1 checklist items have their evidence', all3 and len(items) > 0)

# ---- Item 4: portal (manual) ----
print('== Item 4: portal submission')
if SUBMITTED_ON_PORTAL:
    print('PASS - submitted on the portal as project-5 (flag set by Ayhan)')
else:
    print('TODO (manual): submit as project-5 before Mon 30 Nov 2026 18:00')

== Item 1: README
PASS - README section: overview
PASS - README section: what is where
PASS - README section: how to view the dashboard
PASS - README section: how to run the poc
PASS - README mentions: Spreekorb
PASS - README mentions: UC1
PASS - README mentions: UC2
PASS - README mentions: UC3
PASS - README mentions: black box
PASS - README mentions: SLA
PASS - README mentions: CSAT
PASS - README mentions: synthetic
PASS - README mentions: Team level
PASS - README mentions: minimum group of 5
PASS - README mentions: .env.example
PASS - Tableau Public link in README
PASS - every repo path in README exists (37 checked)
== Item 2: no secrets
PASS - no key patterns in 84 text files
PASS - .env is listed in .gitignore
PASS - .env.example has names only (empty or placeholder values)
PASS - .env.example has the names found in notebooks/ and n8n/
PASS - requirements.txt has every third-party import
== Item 3: Round 1 checklist
Checklist candidates: ['board.md', 'board.md']
NOTE: Round 1 check

/var/folders/v2/tdj6p3nj1bz3h2g88l897pr40000gn/T/ipykernel_50242/1846835538.py:106: FutureWarning: Possible nested set at position 14
  if re.match('^[ ]*[-*][ ]*[[][ xX][]]', l):


In [5]:
# 5 Unchanged: every file outside README.md, requirements.txt, .env.example must have the same hash as before
after = snapshot()
changed = [p for p in HASH_BEFORE if after.get(p) != HASH_BEFORE[p]]
new = [p for p in after if p not in HASH_BEFORE]
ok('unchanged: ' + str(len(HASH_BEFORE)) + ' files (data/processed, dashboard, n8n, langsmith, cost_estimation, other notebooks ...)', not changed)
if changed:
    print('  changed or removed:', changed)
if new:
    print('NOTE - new files appeared during the run:', new)
for d in ['data/processed', 'dashboard', 'n8n', 'langsmith', 'cost_estimation', 'notebooks']:
    n = len([p for p in HASH_BEFORE if p.startswith(d + '/')])
    ok('checked ' + d + ' (' + str(n) + ' files)', n > 0)

PASS - unchanged: 81 files (data/processed, dashboard, n8n, langsmith, cost_estimation, other notebooks ...)
PASS - checked data/processed (8 files)
PASS - checked dashboard (11 files)
PASS - checked n8n (7 files)
PASS - checked langsmith (7 files)
PASS - checked cost_estimation (2 files)
PASS - checked notebooks (9 files)


In [ ]:
# 6 Undo: restores the backup, only if RUN_UNDO = True
RUN_UNDO = False
if RUN_UNDO:
    for p in OUTPUTS:
        if EXISTED[p]:
            shutil.copy2(BK / p, REPO / p)   # put the old file back
        else:
            (REPO / p).unlink(missing_ok=True)   # file did not exist before
    print('Undo done from', BK)
else:
    print('Undo skipped (RUN_UNDO = False)')

In [6]:
# 7 Git: print the commands, never run them (Ayhan commits)
print('git add README.md requirements.txt .env.example notebooks/R1-07_submit.ipynb')
print('git commit -m "R1-07 README, requirements, env example"')

git add README.md requirements.txt .env.example notebooks/R1-07_submit.ipynb
git commit -m "R1-07 README, requirements, env example"
